In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan wordcloud

In [ ]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import pandas as pd
import re


In [ ]:

splits = {'train': 'eng/train-00000-of-00001.parquet', 'test': 'eng/test-00000-of-00001.parquet', 'eval': 'eng/eval-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/dsfsi/vukuzenzele-monolingual/" + splits["train"])
#some data cleaning either we remove the number of empty texts
empty_text_rows = df[
    df["text"].isnull() |
    (df["text"].astype(str).str.strip() == "")
]
print("Number of Rows Empty:", len(empty_text_rows))
print(empty_text_rows)

Number of Rows Empty: 15
                                                 title                 author  \
0                  Young people are SA's real jewels\n  vukuzenzele unnamed\n   
6    The Struggle to give South Africa a more human...  vukuzenzele unnamed\n   
9    No quick fix for load shedding but real progre...  vukuzenzele unnamed\n   
11         Quality affordable smart phone for Africa\n  vukuzenzele unnamed\n   
13        Health’s fight against cancer goes nuclear\n  vukuzenzele unnamed\n   
18                          Land claim reaps rewards\n     Hlengiwe Ngobese\n   
22     New approach to fight corruption gets results\n  vukuzenzele unnamed\n   
27   First time voter eager to register for 2019 el...      More Matshediso\n   
31                          Drawing roads in the sky\n      More Matshediso\n   
36               Vulekamali empowers South Africans \n      More Matshediso\n   
39      Alcohol may damage your unborn baby’s health\n  vukuzenzele unnamed\n   
59 

In [ ]:
df = df.dropna(subset=["text"])

df = df[
    df["text"].astype(str).str.strip() != ""
]

text = df["text"].astype(str).tolist()

In [ ]:
len(text)

105

In [ ]:
def clean_text(text_to_be_cleaned):
  text_to_be_cleaned = text_to_be_cleaned.lower()
  text_to_be_cleaned = re.sub(r"[^\w\s\-]", "",  text_to_be_cleaned)
  text_to_be_cleaned = re.sub(r'\s+', ' ', text_to_be_cleaned)
  return text_to_be_cleaned.strip()
docs = [clean_text(t) for t in text]

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
embeddings = embed_model.encode(
    docs,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=16
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

candidate_labels = [
    "Politics",
    "Health",
    "Economy",
    "Education",
    "Crime",
    "Sports",
    "Technology",
    "Environment",
    "Entertainment",
    "World News",
    "Other"
]

In [ ]:
def classify_topic(topic_words):
    text = ", ".join([w for w, _ in topic_words[:10]])
    result = classifier(text, candidate_labels)
    return result["labels"][0]

In [ ]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from bertopic import BERTopic

model_topic =  BERTopic(
    embedding_model=embed_model,
    vectorizer_model=CountVectorizer(
        stop_words="english",
        ngram_range=(1,3),
        min_df=2,
        max_df=0.85, # as we have 120 articles in the english set
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]+\b"
    ),
    umap_model= UMAP(
        n_neighbors=5,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model= HDBSCAN(
        min_cluster_size=3,
        min_samples=2,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    ),
    nr_topics="auto",
    calculate_probabilities=True,
    representation_model= {
        "main": KeyBERTInspired(),
        "mmr": MaximalMarginalRelevance(diversity=0.7)
    },
    verbose=True
)
topics, probabilities = model_topic.fit_transform(docs, embeddings)

In [ ]:
topics = model_topic.reduce_outliers(docs, topics)
topics = model_topic.reduce_topics(docs, nr_topics=8)
df["topic_id"] = topics
df["confidence"] = probabilities.max(axis=1)

outliers = df[df["topic_id"] == -1]

df = df[df["confidence"] > 0.3]

In [ ]:
print("Number of outliers:", len(outliers))

outlier_ratio = len(outliers) / len(df)
print(f"Outlier ratio: {outlier_ratio:.2%}")

In [ ]:
topic_info = model_topic.get_topic_info()
print(topic_info)

In [ ]:
topic_to_category = {}

for topic_id in topic_info["Topic"]:
    if topic_id != -1:
        words = model_topic.get_topic(topic_id)
        category = classify_topic(words)
        topic_to_category[topic_id] = category

df["news_category"] = df["topic_id"].map(topic_to_category)

In [ ]:
topic_labels = {
    row["Topic"]: ", ".join(row["Representation"][:3])
    for _, row in topic_info.iterrows()
}

df["topic_name"] = df["topic_id"].map(topic_labels)

In [ ]:
for topic_id in topic_info["Topic"].tolist():
    if topic_id != -1:
        words = dict(model_topic.get_topic(topic_id))
        wc = WordCloud(background_color="white").generate_from_frequencies(words)

        plt.figure()
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"Topic {topic_id}")
        plt.show()



In [ ]:
model_topic.visualize_topics()

In [ ]:
model_topic.visualize_barchart()

In [ ]:
model_topic.visualize_hierarchy()

In [ ]:
model_topic.visualize_documents(docs, embeddings=embeddings)

In [ ]:
# =========================
category_counts = df["news_category"].value_counts()

plt.figure(figsize=(10,5))
plt.bar(category_counts.index, category_counts.values)
plt.title("News Category Distribution")
plt.xlabel("Category")
plt.ylabel("Number of Articles")
plt.xticks(rotation=45)
plt.show()


# Detailed Exaplaination and Justification Of Code



# References Used For This Code